In [0]:
%python
df=spark.table("dev.vivek_bronze.sales")

df_final=df.dropDuplicates().dropna().drop("ingestion_date")


df_final.write.mode("overwrite").saveAsTable("dev.vivek_silver.sales")

In [0]:
%sql
create or replace view dev.vivek_gold.top3 as 
(select customer_id, round(sum(total_amount)) as total_amount from dev.vivek_silver.sales group by all order by total_amount desc limit 3)

In [0]:
%sql
select * from dev.naval_gold.top3

In [0]:
%sql
create  view dev.vivek_gold.top3_total as
select dev.vivek_bronze.customers.customer_id, sum(amount) as total from dev.vivek_bronze.customers inner join dev.vivek_bronze.transactions on dev.vivek_bronze.customers.customer_id = dev.vivek_bronze.transactions.customer_id group by dev.vivek_bronze.customers.customer_id order by sum(amount) desc limit 3;

In [0]:
%sql
create table dev.vivek_gold.top3_total_tbl as
select * from dev.vivek_gold.top3_total

In [0]:
%sql
select * from dev.vivek_gold.top3_total_tbl

In [0]:
%sql
create table dev.vivek_silver.monthly_spend_by_category as
select extract(year from transaction_date) as year, extract(month from transaction_date) as month, category, sum(amount) as monthly_spend_by_category from dev.vivek_bronze.customers inner join dev.vivek_bronze.transactions on dev.vivek_bronze.customers.customer_id = dev.vivek_bronze.transactions.customer_id group by all

In [0]:
%sql
create table dev.vivek_gold.agg_cus as 
select sum(amount) as total_amount, dev.vivek_bronze.customers.customer_id from dev.vivek_bronze.customers inner join dev.vivek_bronze.transactions on dev.vivek_bronze.customers.customer_id = dev.vivek_bronze.transactions.customer_id group by dev.vivek_bronze.customers.customer_id;